# Descarga de datos meteorológicos históricos — TFG (versión Colab)
**Fuente:** Open-Meteo Archive API (gratuita, sin API key)

Este notebook **no requiere ningún archivo de entrada**. Descarga los datos directamente
de la API de Open-Meteo y guarda el resultado en `/content/meteo_estaciones_2018_2025.csv`.

**Necesita conexión a internet** (Colab la tiene por defecto).

## Output generado
- `/content/meteo_estaciones_2018_2025.csv` — usado por `integracion_datasets_COLAB.ipynb`

## Cambios respecto al original
- Ruta de salida explícita (`/content/`) para consistencia con el resto del pipeline.
- Descarga automática al ordenador al final.


# Descarga de datos meteorológicos históricos
**Fuente:** Open-Meteo Archive API (gratuita, sin API key)  
**Período:** 2018-01-01 → 2025-12-31  
**Zonas:** 9 municipios con estaciones de esquí (correspondientes al dataset de ocupación hotelera INE)

Variables descargadas por día y zona:
- Temperatura máxima, mínima y media (ºC)
- Precipitación total (mm)
- Nieve caída (cm)
- Velocidad máxima del viento (km/h)

In [ ]:
!pip install requests pandas -q

In [ ]:
import requests
import time
import pandas as pd

# ============================================================
# ESTACIONES: municipio hotelero (INE) -> estación esquí -> coords
# ============================================================
estaciones = [
    {"municipio": "18134 Monachil",          "estacion": "Sierra Nevada",   "lat": 37.094, "lon": -3.394},
    {"municipio": "22054 Benasque",           "estacion": "Cerler",          "lat": 42.606, "lon":  0.572},
    {"municipio": "22125 Huesca",             "estacion": "Zona Huesca",     "lat": 42.136, "lon": -0.408},
    {"municipio": "22130 Jaca",               "estacion": "Candanchu/Astun", "lat": 42.770, "lon": -0.522},
    {"municipio": "22204 Sallent de Gallego", "estacion": "Formigal",        "lat": 42.737, "lon": -0.359},
    {"municipio": "25025 Naut Aran",          "estacion": "Baqueira Beret",  "lat": 42.700, "lon":  0.933},
    {"municipio": "25043 Vall de Boi, La",    "estacion": "Boi Taull",       "lat": 42.490, "lon":  0.852},
    {"municipio": "25120 Lleida",             "estacion": "La Molina",       "lat": 42.345, "lon":  1.987},
    {"municipio": "25243 Vielha e Mijaran",   "estacion": "Val d'Aran",      "lat": 42.691, "lon":  0.795},
]

BASE_URL = "https://archive-api.open-meteo.com/v1/archive"
results = []

for est in estaciones:
    params = {
        "latitude":  est["lat"],
        "longitude": est["lon"],
        "start_date": "2018-01-01",
        "end_date":   "2025-12-31",
        "daily": "temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,snowfall_sum,wind_speed_10m_max",
        "wind_speed_unit": "kmh",
        "timezone": "Europe/Madrid",
    }
    try:
        r = requests.get(BASE_URL, params=params, timeout=30)
        data = r.json()
        if "daily" in data:
            df = pd.DataFrame(data["daily"])
            df["municipio"] = est["municipio"]
            df["estacion"]  = est["estacion"]
            results.append(df)
            print(f"OK  {est['estacion']}: {len(df)} dias descargados")
        else:
            print(f"ERROR {est['estacion']}: {data.get('reason', data)}")
    except Exception as e:
        print(f"ERROR {est['estacion']}: {e}")
    time.sleep(0.6)

print("\nDescarga completada.")

In [ ]:
# ============================================================
# CONSOLIDAR, LIMPIAR Y EXPORTAR
# ============================================================
df_final = pd.concat(results, ignore_index=True)
df_final.rename(columns={"time": "fecha"}, inplace=True)
df_final["fecha"] = pd.to_datetime(df_final["fecha"])

df_final = df_final[[
    "fecha", "municipio", "estacion",
    "temperature_2m_max", "temperature_2m_min", "temperature_2m_mean",
    "precipitation_sum", "snowfall_sum", "wind_speed_10m_max"
]]
df_final.columns = [
    "fecha", "municipio", "estacion",
    "temp_max_c", "temp_min_c", "temp_media_c",
    "precipitacion_mm", "nieve_cm", "viento_max_kmh"
]

print(f"Total registros: {len(df_final):,}")
print(f"Estaciones: {df_final['estacion'].nunique()}")
print(f"Periodo: {df_final['fecha'].min()} --> {df_final['fecha'].max()}")
print(f"\nNulos por columna:")
print(df_final.isnull().sum())
print()
df_final.head()

In [ ]:
# ============================================================
# GUARDAR CSV
# ============================================================
OUTPUT = "/content/meteo_estaciones_2018_2025.csv"
df_final.to_csv(OUTPUT, index=False, encoding="utf-8-sig")
print(f"Guardado: {OUTPUT}")

# Descargar el archivo en Colab
from google.colab import files
files.download(OUTPUT)